# End-to-End LLM Training Tutorial (Offline)

This notebook demonstrates a full toy workflow using **this repo's code only**:
1. Train a tokenizer
2. Pretrain a tiny GPT model on `data/toy`
3. Evaluate perplexity + toy probes
4. Run a tiny SFT pass
5. (Optional) Run tiny DPO updates
6. Load the final checkpoint and generate text

> This is a small educational run. Quality will be limited.


## 0) Setup

- Verifies working directory
- Imports repo modules
- Prints `torch` + `tokenizers` versions
- Creates `artifacts/` and `runs/` paths


In [ ]:
from __future__ import annotations

import json
import random
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from omegaconf import OmegaConf
from tokenizers import __version__ as tokenizers_version
from tokenizers import ByteLevelBPETokenizer
from torch.utils.data import DataLoader

from llmstack.data.datamodule import build_dataloaders
from llmstack.eval.perplexity import evaluate_perplexity
from llmstack.eval.probes import (
    bracket_matching_probe,
    repeat_after_me_probe,
    simple_addition_probe,
)
from llmstack.model.gpt import GPTModel
from llmstack.optim.adamw import build_adamw
from llmstack.optim.schedulers import build_scheduler
from llmstack.posttrain.dpo_trainer import DPOTrainer
from llmstack.posttrain.preference_dataset import PreferenceDataset
from llmstack.posttrain.sft_trainer import SFTDataset
from llmstack.tokenization.tokenizer import TokenizerWrapper
from llmstack.train.checkpointing import (
    build_train_state,
    load_checkpoint,
    save_checkpoint,
)
from llmstack.train.engine import Trainer
from llmstack.utils.logging import RunLogger
from llmstack.utils.seed import seed_everything


In [ ]:
# Verify repo-relative paths.
repo_root = Path.cwd()
assert (repo_root / "data" / "toy" / "train.jsonl").exists(), "Run notebook from repo root (llm-from-scratch-stack/)."

print("Working directory:", repo_root)
print("torch:", torch.__version__)
print("tokenizers:", tokenizers_version)

artifacts_dir = repo_root / "artifacts"
runs_dir = repo_root / "runs"
artifacts_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)

run_name = f"notebook_{datetime.now():%Y%m%d_%H%M%S}"
run_dir = runs_dir / run_name
run_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir = run_dir / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

print("Run directory:", run_dir)


## 1) Reproducibility + Device/Precision

Single-process, rank-0 execution only. We select precision automatically:
- CUDA + bf16 support -> `bf16`
- CUDA otherwise -> `fp16`
- CPU -> `fp32`


In [ ]:
SEED = 1234
seed_everything(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
    PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "fp16"
else:
    DEVICE = "cpu"
    PRECISION = "fp32"

print({"device": DEVICE, "precision": PRECISION, "seed": SEED})


## 2) Train a Byte-Level BPE Tokenizer

We train on `data/toy/train.jsonl` and save artifacts to `artifacts/tokenizer_toy/`.


In [ ]:
tokenizer_dir = artifacts_dir / "tokenizer_toy"
tokenizer_dir.mkdir(parents=True, exist_ok=True)
corpus_txt = tokenizer_dir / "corpus.txt"

with (repo_root / "data" / "toy" / "train.jsonl").open("r", encoding="utf-8") as f, corpus_txt.open("w", encoding="utf-8") as w:
    for line in f:
        row = json.loads(line)
        w.write(row["text"] + "
")

bpe = ByteLevelBPETokenizer()
bpe.train(
    files=[str(corpus_txt)],
    vocab_size=2048,
    min_frequency=2,
    special_tokens=["<pad>", "<bos>", "<eos>", "<unk>"],
)
bpe.save_model(str(tokenizer_dir))
bpe.save(str(tokenizer_dir / "tokenizer.json"))

tokenizer = TokenizerWrapper.from_dir(str(tokenizer_dir))
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Tokenizer files:", sorted([p.name for p in tokenizer_dir.iterdir()]))


## 3) Build Tiny Pretraining Config + DataLoaders

Defaults are intentionally small to keep runtime modest.


In [ ]:
# Runtime knobs (adjust to your hardware)
PRETRAIN_STEPS = 80      # suggested 50-200
SFT_STEPS = 30           # suggested 20-50
DPO_STEPS = 15           # suggested 10-30
RUN_DPO = False          # set True to enable optional DPO section

cfg = OmegaConf.create(
    {
        "tokenizer_dir": str(tokenizer_dir),
        "project_name": "llm-from-scratch-stack-notebook",
        "use_wandb": False,
        "model": {
            "vocab_size": tokenizer.vocab_size,
            "n_layers": 4,
            "n_heads": 4,
            "d_model": 256,
            "d_ff": 1024,
            "max_seq_len": 128,
            "dropout": 0.1,
            "norm_type": "layernorm",
            "rope_enabled": False,
            "tie_embeddings": True,
        },
        "data": {
            "train_path": "data/toy/train.jsonl",
            "val_path": "data/toy/val.jsonl",
            "format": "jsonl",
            "text_field": "text",
            "seq_len": 128,
            "pack_sequences": True,
            "num_workers": 0,
            "shuffle": True,
            "streaming": False,
        },
        "train": {
            "seed": SEED,
            "device": DEVICE,
            "precision": PRECISION,
            "compile": False,
            "grad_clip": 1.0,
            "grad_accum_steps": 1,
            "micro_batch_size": 4,
            "max_steps": PRETRAIN_STEPS,
            "eval_interval": 20,
            "log_interval": 10,
            "save_interval": 20,
            "out_dir": str(runs_dir),
            "run_dir": str(run_dir),
            "resume_path": None,
            "max_eval_batches": 20,
        },
        "optim": {
            "name": "adamw",
            "lr": 3e-4,
            "betas": [0.9, 0.95],
            "weight_decay": 0.1,
            "eps": 1e-8,
        },
        "sched": {
            "name": "warmup_cosine",
            "warmup_steps": 20,
            "min_lr": 3e-5,
        },
        "dist": {
            "mode": "single",
            "backend": "gloo",
            "find_unused_params": False,
            "fsdp_shard_strategy": "FULL_SHARD",
            "fsdp_cpu_offload": False,
            "fsdp_mixed_precision": False,
        },
    }
)

train_loader, val_loader = build_dataloaders(cfg, tokenizer)
print("Train batches/epoch:", len(train_loader), "Val batches/epoch:", len(val_loader))


## 4) Pretrain Tiny GPT


In [ ]:
model = GPTModel(cfg).to(DEVICE)
optimizer = build_adamw(model.parameters(), cfg)
scheduler = build_scheduler(optimizer, cfg)
logger = RunLogger(str(run_dir))

trainer = Trainer(cfg, model, optimizer, scheduler, train_loader, val_loader, logger, DEVICE)
trainer.fit()
logger.close()

pretrain_last_ckpt = checkpoints_dir / "last.pt"
state = build_train_state(model, optimizer, scheduler, trainer.scaler, trainer.step, trainer.best_val, cfg)
save_checkpoint(str(pretrain_last_ckpt), state)

# Lightweight export (weights + tiny manifest)
export_dir = run_dir / "export_pretrain"
export_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), export_dir / "model_state_dict.pt")
(export_dir / "manifest.json").write_text(
    json.dumps({"tokenizer_dir": str(tokenizer_dir), "step": trainer.step}, indent=2),
    encoding="utf-8",
)

print("Saved pretrain checkpoint:", pretrain_last_ckpt)
print("Saved export dir:", export_dir)


## 5) Evaluate: Perplexity + Offline Probes


In [ ]:
def run_eval_report(model_obj, val_loader_obj, tokenizer_obj, device, report_path: Path, max_batches: int = 20):
    report = evaluate_perplexity(model_obj, val_loader_obj, device, max_batches=max_batches)
    report.update(repeat_after_me_probe(model_obj, tokenizer_obj, device))
    report.update(bracket_matching_probe(model_obj, tokenizer_obj, device))
    report.update(simple_addition_probe(model_obj, tokenizer_obj, device))
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    return report

pretrain_eval_report = run_eval_report(
    model, val_loader, tokenizer, DEVICE, run_dir / "eval_pretrain.json", max_batches=20
)
print(json.dumps(pretrain_eval_report, indent=2))


## 6) Tiny SFT Example

We create a tiny prompt/response dataset and run short supervised fine-tuning.


In [ ]:
sft_path = repo_root / "data" / "toy" / "sft.jsonl"
sft_rows = [
    {"prompt": "User: say hello politely", "response": "Assistant: Hello! It's great to meet you."},
    {"prompt": "User: what is 2 + 2?", "response": "Assistant: 2 + 2 = 4."},
    {"prompt": "User: finish: roses are", "response": "Assistant: red."},
]
sft_path.write_text("
".join(json.dumps(r) for r in sft_rows) + "
", encoding="utf-8")
print("Wrote:", sft_path)


In [ ]:
sft_run_dir = runs_dir / f"{run_name}_sft"
sft_run_dir.mkdir(parents=True, exist_ok=True)
(cfg.train.run_dir,) = (str(sft_run_dir),)
cfg.train.max_steps = SFT_STEPS
cfg.train.eval_interval = max(10, SFT_STEPS // 2)
cfg.train.save_interval = max(10, SFT_STEPS // 2)

sft_model = GPTModel(cfg).to(DEVICE)
pretrain_ckpt = load_checkpoint(str(pretrain_last_ckpt), map_location=DEVICE)
sft_model.load_state_dict(pretrain_ckpt["model"])

sft_ds = SFTDataset(str(sft_path), tokenizer, cfg.data.seq_len, "
### Response:
")
sft_train_loader = DataLoader(sft_ds, batch_size=cfg.train.micro_batch_size, shuffle=True)
sft_val_loader = DataLoader(sft_ds, batch_size=cfg.train.micro_batch_size, shuffle=False)

sft_optimizer = build_adamw(sft_model.parameters(), cfg)
sft_scheduler = build_scheduler(sft_optimizer, cfg)
sft_logger = RunLogger(str(sft_run_dir))

sft_trainer = Trainer(
    cfg, sft_model, sft_optimizer, sft_scheduler, sft_train_loader, sft_val_loader, sft_logger, DEVICE
)
sft_trainer.fit()
sft_logger.close()

sft_ckpt_path = sft_run_dir / "checkpoints" / "last.pt"
sft_ckpt_path.parent.mkdir(parents=True, exist_ok=True)
save_checkpoint(
    str(sft_ckpt_path),
    build_train_state(
        sft_model,
        sft_optimizer,
        sft_scheduler,
        sft_trainer.scaler,
        sft_trainer.step,
        sft_trainer.best_val,
        cfg,
    ),
)

sft_eval_report = run_eval_report(
    sft_model, val_loader, tokenizer, DEVICE, sft_run_dir / "eval_sft.json", max_batches=20
)
print("SFT checkpoint:", sft_ckpt_path)
print(json.dumps(sft_eval_report, indent=2))


## 7) Optional Tiny DPO Example

Set `RUN_DPO = True` to execute.


In [ ]:
dpo_run_dir = runs_dir / f"{run_name}_dpo"
dpo_eval_report = None

if RUN_DPO:
    dpo_run_dir.mkdir(parents=True, exist_ok=True)
    prefs_path = repo_root / "data" / "toy" / "prefs.jsonl"
    prefs_rows = [
        {
            "prompt": "User: summarize -> cats are playful and curious.",
            "chosen": " Assistant: Cats are playful, curious animals.",
            "rejected": " Assistant: The weather is made of triangles.",
        },
        {
            "prompt": "User: answer politely: thanks",
            "chosen": " Assistant: You're welcome! Happy to help.",
            "rejected": " Assistant: No.",
        },
    ]
    prefs_path.write_text("
".join(json.dumps(r) for r in prefs_rows) + "
", encoding="utf-8")

    dpo_model = GPTModel(cfg).to(DEVICE)
    sft_ckpt = load_checkpoint(str(sft_ckpt_path), map_location=DEVICE)
    dpo_model.load_state_dict(sft_ckpt["model"])

    dpo_trainer = DPOTrainer(dpo_model, beta=0.1)
    dpo_optimizer = torch.optim.AdamW(dpo_model.parameters(), lr=cfg.optim.lr)
    pref_ds = PreferenceDataset(str(prefs_path))

    for step in range(DPO_STEPS):
        row = pref_ds[step % len(pref_ds)]
        chosen_ids = torch.tensor([tokenizer.encode(row["prompt"] + row["chosen"])], device=DEVICE)
        rejected_ids = torch.tensor([tokenizer.encode(row["prompt"] + row["rejected"])], device=DEVICE)
        loss = dpo_trainer.loss(chosen_ids, rejected_ids)
        dpo_optimizer.zero_grad(set_to_none=True)
        loss.backward()
        dpo_optimizer.step()

    dpo_ckpt_path = dpo_run_dir / "checkpoints" / "last.pt"
    dpo_ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    save_checkpoint(
        str(dpo_ckpt_path),
        {
            "model": dpo_model.state_dict(),
            "optimizer": dpo_optimizer.state_dict(),
            "step": DPO_STEPS,
            "config": OmegaConf.to_container(cfg, resolve=True),
        },
    )

    dpo_eval_report = run_eval_report(
        dpo_model, val_loader, tokenizer, DEVICE, dpo_run_dir / "eval_dpo.json", max_batches=20
    )
    print("DPO checkpoint:", dpo_ckpt_path)
    print(json.dumps(dpo_eval_report, indent=2))
else:
    print("Skipping DPO. Set RUN_DPO=True above to enable this section.")


## 8) Inference Demo (Greedy Decode)

We load the final available checkpoint in this priority:
`DPO (if run) -> SFT -> Pretrain`, then generate a short completion.


In [ ]:
def greedy_generate(model_obj, tok, prompt: str, device: str, max_new_tokens: int = 40):
    model_obj.eval()
    ids = tok.encode(prompt)
    x = torch.tensor([ids], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            out = model_obj(x)
            next_id = int(out["logits"][0, -1].argmax().item())
            x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
            if next_id == tok.eos_id or x.shape[1] >= cfg.model.max_seq_len:
                break
    return tok.decode(x[0].tolist())

# Choose final model
final_model = GPTModel(cfg).to(DEVICE)
if RUN_DPO and (dpo_run_dir / "checkpoints" / "last.pt").exists():
    final_ckpt = load_checkpoint(str(dpo_run_dir / "checkpoints" / "last.pt"), map_location=DEVICE)
elif sft_ckpt_path.exists():
    final_ckpt = load_checkpoint(str(sft_ckpt_path), map_location=DEVICE)
else:
    final_ckpt = load_checkpoint(str(pretrain_last_ckpt), map_location=DEVICE)

final_model.load_state_dict(final_ckpt["model"])

prompt = "Sample line:"
generated = greedy_generate(final_model, tokenizer, prompt, DEVICE, max_new_tokens=40)
print("Prompt:", prompt)
print("Generated:", generated)


## 9) Where outputs are written

- Tokenizer artifacts: `artifacts/tokenizer_toy/`
- Pretrain run: `runs/<timestamp>/`
- SFT run: `runs/<timestamp>_sft/`
- Optional DPO run: `runs/<timestamp>_dpo/`
- Evaluation reports: `eval_*.json` in each run directory
